In [16]:
# Step 0: Mount Drive and set base path (needed before saving documentation)

from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/Quantum_DL_MNIST"
print("Base path set:", base)

Mounted at /content/drive
Base path set: /content/drive/MyDrive/Quantum_DL_MNIST


In [1]:
# Step 1: Install and import PennyLane

!pip install pennylane pennylane-lightning -q

import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt

print("PennyLane version:", qml.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.1 MB/s eta 0:00:00
PennyLane version: 0.45.1


In [2]:
# Step 2: Create a 1-qubit device and look at the simplest possible circuit
# A qubit starts in state |0> - measuring it should always give 0

dev1 = qml.device("default.qubit", wires=1)

@qml.qnode(dev1)
def do_nothing_circuit():
    return qml.expval(qml.PauliZ(0))

result = do_nothing_circuit()
print("Expectation value of Z on an untouched qubit:", result)
print("(+1 means the qubit is in state |0>, -1 would mean state |1>)")

Expectation value of Z on an untouched qubit: 1.0
(+1 means the qubit is in state |0>, -1 would mean state |1>)


In [3]:
# Step 3: Apply a Hadamard gate - puts the qubit into SUPERPOSITION
# (equal mix of |0> and |1> at the same time - a purely quantum idea)

@qml.qnode(dev1)
def superposition_circuit():
    qml.Hadamard(wires=0)
    return qml.expval(qml.PauliZ(0))

result = superposition_circuit()
print("Expectation value of Z after Hadamard:", result)
print("(0 means the qubit is EQUALLY likely to be measured as 0 or 1)")

# Let's also look at the actual probabilities, not just the expectation value
@qml.qnode(dev1)
def superposition_probs():
    qml.Hadamard(wires=0)
    return qml.probs(wires=0)

probs = superposition_probs()
print("\nProbability of measuring 0:", probs[0])
print("Probability of measuring 1:", probs[1])

Expectation value of Z after Hadamard: 0.0
(0 means the qubit is EQUALLY likely to be measured as 0 or 1)

Probability of measuring 0: 0.4999999999999999
Probability of measuring 1: 0.4999999999999999


In [4]:
# Step 4: RY rotation - this is how CLASSICAL data gets encoded into
# the quantum circuit in Chunk 7 (Section 8.1 - "RY(x0), RY(x1)...")
# RY(theta) rotates the qubit by angle theta - theta=0 keeps it at |0>,
# theta=pi flips it fully to |1>, anything between is a mix

import numpy as np

dev1 = qml.device("default.qubit", wires=1)

@qml.qnode(dev1)
def ry_circuit(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

angles = [0, np.pi/4, np.pi/2, np.pi]
for theta in angles:
    result = ry_circuit(theta)
    print(f"RY({theta:.4f}) -> Z expectation = {result:.4f}")

print("\nNotice: as theta goes from 0 to pi, the expectation value")
print("smoothly moves from +1 (state |0>) to -1 (state |1>)")
print("This is EXACTLY how our CNN's 4 features get encoded as quantum")
print("rotation angles in Chunk 7 (after being squashed to [-pi, pi] by tanh)")

RY(0.0000) -> Z expectation = 1.0000
RY(0.7854) -> Z expectation = 0.7071
RY(1.5708) -> Z expectation = 0.0000
RY(3.1416) -> Z expectation = -1.0000

Notice: as theta goes from 0 to pi, the expectation value
smoothly moves from +1 (state |0>) to -1 (state |1>)
This is EXACTLY how our CNN's 4 features get encoded as quantum
rotation angles in Chunk 7 (after being squashed to [-pi, pi] by tanh)


In [5]:
# Step 5: RZ rotation - the SECOND rotation used in Chunk 7's circuit
# RZ rotates around a different axis - it doesn't change Z-measurement
# probabilities by itself, but it changes RELATIVE PHASE, which matters
# once you combine it with RY and entanglement (next step)

dev1 = qml.device("default.qubit", wires=1)

@qml.qnode(dev1)
def ry_then_rz_circuit(theta_y, theta_z):
    qml.RY(theta_y, wires=0)
    qml.RZ(theta_z, wires=0)
    return qml.expval(qml.PauliZ(0))

# Notice: changing RZ alone (theta_y fixed) does NOT change Z expectation
print("Fixing RY, varying RZ:")
for theta_z in [0, np.pi/2, np.pi]:
    result = ry_then_rz_circuit(np.pi/4, theta_z)
    print(f"  RY(pi/4) + RZ({theta_z:.4f}) -> Z expectation = {result:.4f}")

print("\nThis is expected - RZ alone doesn't move the Z-axis measurement.")
print("But RZ becomes important once qubits are ENTANGLED with each other")
print("(next step) - that's why Chunk 7 uses BOTH RY and RZ, not RY alone")
print("(Section 8.2 of the protocol explains this exact design choice)")

Fixing RY, varying RZ:
  RY(pi/4) + RZ(0.0000) -> Z expectation = 0.7071
  RY(pi/4) + RZ(1.5708) -> Z expectation = 0.7071
  RY(pi/4) + RZ(3.1416) -> Z expectation = 0.7071

This is expected - RZ alone doesn't move the Z-axis measurement.
But RZ becomes important once qubits are ENTANGLED with each other
(next step) - that's why Chunk 7 uses BOTH RY and RZ, not RY alone
(Section 8.2 of the protocol explains this exact design choice)


In [6]:
# Step 6: CNOT gate - creates ENTANGLEMENT between two qubits
# CNOT flips the second (target) qubit ONLY IF the first (control) qubit is |1>
# This links the qubits together - measuring one affects what you'd expect from the other

dev2 = qml.device("default.qubit", wires=2)

@qml.qnode(dev2)
def entanglement_demo():
    qml.Hadamard(wires=0)      # put qubit 0 into superposition
    qml.CNOT(wires=[0, 1])     # entangle qubit 0 (control) with qubit 1 (target)
    return qml.probs(wires=[0, 1])

probs = entanglement_demo()
print("Probabilities of each 2-qubit outcome:")
print(f"  |00> : {probs[0]:.4f}")
print(f"  |01> : {probs[1]:.4f}")
print(f"  |10> : {probs[2]:.4f}")
print(f"  |11> : {probs[3]:.4f}")

print("\nNotice: ONLY |00> and |11> have probability - never |01> or |10>")
print("The two qubits are now CORRELATED - if you measure qubit 0 as 0,")
print("qubit 1 is GUARANTEED to also be 0. Measure qubit 0 as 1, qubit 1")
print("is GUARANTEED to be 1 too. Neither qubit has a value on its own")
print("anymore - this is entanglement (Section 9, Level 2 of the protocol)")

Probabilities of each 2-qubit outcome:
  |00> : 0.5000
  |01> : 0.0000
  |10> : 0.0000
  |11> : 0.5000

Notice: ONLY |00> and |11> have probability - never |01> or |10>
The two qubits are now CORRELATED - if you measure qubit 0 as 0,
qubit 1 is GUARANTEED to also be 0. Measure qubit 0 as 1, qubit 1
is GUARANTEED to be 1 too. Neither qubit has a value on its own
anymore - this is entanglement (Section 9, Level 2 of the protocol)


In [9]:
# Step 7: The clean, standard way to SEE a phase
# rotation's effect - Hadamard, then RZ, then Hadamard again, then measure.
# This "sandwich" trick converts an invisible phase into a visible,
# measurable change in Z-expectation. No entanglement needed for this part.

dev1 = qml.device("default.qubit", wires=1)

@qml.qnode(dev1)
def phase_visibility_demo(theta_z):
    qml.Hadamard(wires=0)     # create superposition
    qml.RZ(theta_z, wires=0)  # apply phase - invisible to Z measurement so far
    qml.Hadamard(wires=0)     # convert phase back into a Z-measurable difference
    return qml.expval(qml.PauliZ(0))

print("Z-expectation after Hadamard -> RZ(theta) -> Hadamard:")
for theta_z in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    result = phase_visibility_demo(theta_z)
    print(f"  RZ({theta_z:.4f}) -> Z expectation = {result:.4f}")

print("\nNOW theta clearly and smoothly changes the result (this follows")
print("cos(theta) exactly). This confirms RZ genuinely does something -")
print("it just doesn't show up in a single, isolated Z measurement without")
print("something (another rotation, or entanglement + a joint measurement)")
print("to convert that phase into something Z can detect.")

Z-expectation after Hadamard -> RZ(theta) -> Hadamard:
  RZ(0.0000) -> Z expectation = 1.0000
  RZ(1.5708) -> Z expectation = 0.0000
  RZ(3.1416) -> Z expectation = -1.0000
  RZ(4.7124) -> Z expectation = -0.0000
  RZ(6.2832) -> Z expectation = 1.0000

NOW theta clearly and smoothly changes the result (this follows
cos(theta) exactly). This confirms RZ genuinely does something -
it just doesn't show up in a single, isolated Z measurement without
something (another rotation, or entanglement + a joint measurement)
to convert that phase into something Z can detect.


In [10]:
# Step 8: Build the ACTUAL circuit architecture from Chunk 7 (Section 8.1)
# 4 qubits, RY data encoding, 2 layers of (RY+RZ rotations + ring CNOT),
# then measure all 4 qubits' Z expectation values

n_qubits = 4
dev4 = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev4)
def full_circuit_demo(inputs, weights):
    # --- Data encoding: RY(x0), RY(x1), RY(x2), RY(x3) ---
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    # --- Layer 1: RY(theta) + RZ(phi) per qubit, then RING CNOT ---
    for i in range(n_qubits):
        qml.RY(weights[0][i][0], wires=i)
        qml.RZ(weights[0][i][1], wires=i)
    for i in range(n_qubits):
        qml.CNOT(wires=[i, (i + 1) % n_qubits])   # ring: 0->1->2->3->0

    # --- Layer 2: same pattern again ---
    for i in range(n_qubits):
        qml.RY(weights[1][i][0], wires=i)
        qml.RZ(weights[1][i][1], wires=i)
    for i in range(n_qubits):
        qml.CNOT(wires=[i, (i + 1) % n_qubits])

    # --- Measurement: <Z0>, <Z1>, <Z2>, <Z3> ---
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("Full circuit defined (matches Chunk 7 / Section 8.1 exactly)")

Full circuit defined (matches Chunk 7 / Section 8.1 exactly)


In [12]:
# Step 9: Try the circuit with some example inputs and random weights,
# just to confirm it runs and produces 4 output values

np.random.seed(42)

# Example: 4 "features" already squashed to [-pi, pi] (this is what our
# CNN backbone's 4-dim output will look like after the tanh scaling in Chunk 7)
example_inputs = np.array([0.5, -1.2, 2.0, -0.3])

# 2 layers x 4 qubits x 2 rotations (RY, RZ) = 16 trainable quantum parameters
example_weights = np.random.uniform(-0.1, 0.1, size=(2, 4, 2))

output = full_circuit_demo(example_inputs, example_weights)
print("Circuit inputs (4 features):", example_inputs)
print("Circuit output (4 Z-expectation values):", output)

print(f"\nTotal trainable quantum parameters: {example_weights.size}")
print("(This matches Section 8.1: 4 qubits x 2 rotations x 2 layers = 16)")
print(qml.draw(full_circuit_demo)(example_inputs, example_weights))

Circuit inputs (4 features): [ 0.5 -1.2  2.  -0.3]
Circuit output (4 Z-expectation values): [np.float64(0.2517194089102317), np.float64(-0.2898189146381983), np.float64(0.30244860727556155), np.float64(-0.3043307808309286)]

Total trainable quantum parameters: 16
(This matches Section 8.1: 4 qubits x 2 rotations x 2 layers = 16)
0: ──RY(0.50)───RY(-0.03)──RZ(0.09)──╭●───────╭X──RY(0.02)───RZ(0.04)──╭●───────╭X─┤  <Z>
1: ──RY(-1.20)──RY(0.05)───RZ(0.02)──╰X─╭●────│───RY(-0.10)──RZ(0.09)──╰X─╭●────│──┤  <Z>
2: ──RY(2.00)───RY(-0.07)──RZ(-0.07)────╰X─╭●─│───RY(0.07)───RZ(-0.06)────╰X─╭●─│──┤  <Z>
3: ──RY(-0.30)──RY(-0.09)──RZ(0.07)────────╰X─╰●──RY(-0.06)──RZ(-0.06)───────╰X─╰●─┤  <Z>


In [17]:
# Step 11: Documentation sketch for Quantum Fundamentals Primer (Chunk 4)

quantum_primer_documentation = {
    "notebook_purpose": "Conceptual primer before building the real hybrid model in Chunk 7. "
                         "Hands-on demonstrations of the exact gates/concepts used in Section 8.1's "
                         "quantum circuit, verified numerically rather than just asserted.",

    "concepts_demonstrated": {
        "level_1_foundations": "Qubit vs classical bit - measured expectation values, "
                                "state |0> gives Z=+1, state |1> gives Z=-1",
        "level_2_quantum_behavior": {
            "superposition": "Hadamard gate creates exact 50/50 measurement probability",
            "entanglement": "CNOT after Hadamard creates perfectly correlated 2-qubit outcomes "
                             "(only |00> and |11> possible, never |01> or |10>)"
        },
        "level_3_gates": {
            "RY": "Smoothly rotates Z-expectation from +1 to -1 as angle goes 0 to pi - "
                  "this is exactly how CNN features get encoded as rotation angles in Chunk 7",
            "RZ": "Does NOT affect Z-basis measurement directly (phase-only rotation). "
                  "Its effect only becomes visible via a second rotation or different "
                  "measurement basis (demonstrated correctly via Hadamard-RZ-Hadamard sandwich, "
                  "which traces out cos(theta) exactly)",
            "CNOT": "Entangles two qubits - flips target qubit only if control qubit is |1>"
        },
        "level_4_qml": "Full 4-qubit variational circuit assembled and verified - matches "
                        "Section 8.1 exactly: RY encoding, 2x(RY+RZ layer + ring CNOT), "
                        "measure <Z> on all 4 wires. Confirmed 16 trainable parameters "
                        "(4 qubits x 2 rotations x 2 layers)."
    },

    "corrections_made_during_primer": [
        "Initial RZ-with-entanglement demo (Step 7, first attempt) incorrectly measured "
        "qubit 1's Z-expectation while varying qubit 0's RZ - showed no change because "
        "phase effects don't appear in isolated single-qubit Z measurements this way.",
        "Second attempt (X-basis measurement) also failed to isolate the effect correctly "
        "due to tracing out the entangled partner qubit.",
        "Corrected with the standard Hadamard-RZ-Hadamard phase-to-population sandwich, "
        "which cleanly and correctly traces out cos(theta) - textbook-verified technique.",
        "Lesson: always verify claims against actual printed numbers, not just narration - "
        "consistent with Section 25's exploratory, evidence-first framing."
    ],

    "circuit_diagram_text": (
        "0: --RY(x0)--RY(t)--RZ(p)--[ring CNOT]--RY(t)--RZ(p)--[ring CNOT]--<Z>\n"
        "1: --RY(x1)--RY(t)--RZ(p)--[ring CNOT]--RY(t)--RZ(p)--[ring CNOT]--<Z>\n"
        "2: --RY(x2)--RY(t)--RZ(p)--[ring CNOT]--RY(t)--RZ(p)--[ring CNOT]--<Z>\n"
        "3: --RY(x3)--RY(t)--RZ(p)--[ring CNOT]--RY(t)--RZ(p)--[ring CNOT]--<Z>\n"
        "(actual qml.draw() text output saved separately)"
    ),

    "verified_parameter_count": "16 trainable quantum rotation parameters, matches Section 8.1",

    "ready_for_chunk_7": "Yes - all gate-level concepts verified numerically. Chunk 7 will wrap "
                          "this exact circuit structure in qml.qnn.TorchLayer for PyTorch integration, "
                          "add the learnable per-feature scaling + tanh bounding (Section 8.1), "
                          "and connect it to the CNN backbone's 4-feature bottleneck (Chunk 3)."
}

import json
with open(f"{base}/documentation/04_Quantum_Primer_documentation_sketch.json", "w") as f:
    json.dump(quantum_primer_documentation, f, indent=2)

print("Saved documentation sketch to documentation/04_Quantum_Primer_documentation_sketch.json")
print("\n--- CHUNK 4 (Quantum Fundamentals Primer) COMPLETE ---")

Saved documentation sketch to documentation/04_Quantum_Primer_documentation_sketch.json

--- CHUNK 4 (Quantum Fundamentals Primer) COMPLETE ---
